# Google Gemini API 구조 실습

A-PI 사전조사 — Google Gemini(Gemini Developer API, AI Studio 키) 를 7개 항목으로 실측하고 `evidence/gemini/*.json` 을 남기는 노트북.
기준일 2026-10-19 · SDK `google-genai` 2.31.0 · Python 3.12 · 어댑터 `apicompare.providers.gemini_adapter.GeminiAdapter`

- **현재 `.env` 에 `GEMINI_API_KEY` 가 없으므로 기본은 NO_KEY 모드** → 모든 섹션이 `[BLOCKED: no key]` 를 출력하고 evidence 를 `blocked` 로 저장한다.
- 오늘 https://aistudio.google.com/apikey 에서 **무료 키**를 받아 `.env` 에 `GEMINI_API_KEY=...` 한 줄을 추가하고 커널을 재시작하면 그대로 LIVE 로 동작한다(코드 수정 없음).
- 무료 티어(Free tier) 입력은 **Google 제품 개선에 사용될 수 있다**(Paid tier 는 미사용). 기업 샘플 데이터·개인정보는 절대 넣지 않는다 — 이 노트북의 프롬프트는 공개 상식 질문만 쓴다.
- 금지: 키 값 출력, `os.environ` 전체 출력, `print(client)`. 키 존재 여부는 `apicompare.config.keys()` 의 bool 로만 본다.

## 0. 준비

- 공식 방법: `pip install -U google-genai` → `from google import genai; client = genai.Client()` (환경변수 `GEMINI_API_KEY` 또는 `GOOGLE_API_KEY` 자동 인식, 둘 다 있으면 `GOOGLE_API_KEY` 우선). REST 는 헤더 `x-goog-api-key: $GEMINI_API_KEY` (Bearer 아님).
- 구 패키지 `google-generativeai`(`genai.GenerativeModel`) 는 2025-11-30 deprecated — 쓰지 않는다.
- 어댑터는 `HttpOptions(timeout=밀리초, retry_options=HttpRetryOptions(attempts=max_retries+1))` 로 **재시도 0회·timeout 30s** 를 고정한다(§6 에서 그 이유를 소스로 확인).
- 출처: https://ai.google.dev/gemini-api/docs/libraries · https://ai.google.dev/gemini-api/docs/api-key · https://googleapis.github.io/python-genai/

In [ ]:
import sys
assert sys.version_info >= (3, 12), f"Python 3.12+ 필요: {sys.version}"

from apicompare.config import load_env, keys
load_env()  # 프로젝트 루트 .env 로드 (값은 절대 출력하지 않는다)

import google.genai as genai_pkg
print("google-genai", genai_pkg.__version__)

from apicompare import __version__ as apicompare_version
from apicompare.evidence import save_evidence, summary
from apicompare.models import to_dict, NotSupported
from apicompare.providers import get_adapter

adapter = get_adapter("gemini")            # timeout_s=30, max_retries=0, default_model="gemini-3.8-flash"
MODEL = adapter.default_model
NO_KEY = adapter.mode == "NO_KEY"
PROMPT = "대한민국의 수도를 한 문장으로 알려줘."

print(f"apicompare {apicompare_version} · provider={adapter.label} · model={MODEL}")
print("키 존재 여부(bool):", {k: v for k, v in keys().items() if k.startswith("GEMINI")})
print("MODE:", "[NO_KEY] 모든 섹션 → [BLOCKED: no key]" if NO_KEY else "[LIVE] 실측 진행 (Free tier 면 과금 없음)")

def blocked(item: str, extra: dict | None = None):
    """NO_KEY 분기 공통: BLOCKED 출력 + evidence(blocked, note='no key')."""
    print("[BLOCKED: no key]")
    payload = {"error": to_dict(adapter.no_key_error())}
    if extra:
        payload.update(extra)
    return save_evidence("gemini", item, payload, "blocked", note="no key", model=MODEL)

## 1. 모델 목록 조회

- 공식 방법: `GET https://generativelanguage.googleapis.com/v1beta/models` (pageSize 기본 50·최대 1000, pageToken) · 단일 조회 `GET /v1beta/models/{model}`. SDK `client.models.list()`(Pager 가 페이지네이션 처리) / `client.models.get(model=...)`.
- 인증: 표준 Gemini API 키. 추가 권한 불필요. 토큰 과금 없음.
- **3사 중 메타데이터가 가장 풍부**: `name, baseModelId, version, displayName, description, inputTokenLimit, outputTokenLimit, supportedGenerationMethods[](SDK: supported_actions), thinking, temperature, maxTemperature, topP, topK`. → Simulator 모델 카탈로그(컨텍스트 한도 검증)를 API 로 자동 구성 가능. OpenAI 는 context window 미제공, Anthropic 은 max_input_tokens/max_tokens 만.
- 주의: 가격·지식 컷오프·deprecation 날짜는 **없음**(pricing/deprecations 페이지 참고). 2.5 세대는 deprecated 가 아니어도 '과거 사용자만 접근 가능' → 목록에 보여도 호출이 거부될 수 있음. gemini-3.7-flash·3.5-flash 는 2026-10-08 retire → 3.8/3.6 으로 자동 라우팅.
- 출처: https://ai.google.dev/api/models · https://ai.google.dev/gemini-api/docs/models · https://ai.google.dev/gemini-api/docs/deprecations

In [ ]:
if NO_KEY:
    models = []
    print("[BLOCKED: no key]")
else:
    models = adapter.list_models()
    print(f"{len(models)} models")
    if not models and adapter.last_error:
        print("list_models 실패:", to_dict(adapter.last_error))
    for m in models[:15]:
        print(f"{m.id:40s} ctx={m.context_window} max_out={m.max_output_tokens} thinking={m.capabilities.get('thinking')} actions={m.capabilities.get('supported_actions')}")

In [ ]:
# 원본 객체/필드 확인: SDK Model 은 snake_case (input_token_limit, output_token_limit, supported_actions, thinking, temperature, top_p, top_k)
if NO_KEY:
    print("[BLOCKED: no key]")
else:
    target = next((m for m in models if m.id == MODEL), models[0] if models else None)
    if target:
        print({k: v for k, v in target.raw.items() if k != "description"})
        gen_models = [m for m in models if "generateContent" in m.capabilities.get("supported_actions", [])]
        print(f"generateContent 지원 모델 {len(gen_models)}/{len(models)} · 가격 필드 있음? {'pricing' in target.raw} (없음 → pricing.PRICE_TABLE 로컬 관리)")

관찰 메모: (실행 후 채우기) — 모델 수 / `gemini-3.8-flash` 의 input/output_token_limit / thinking 플래그 / 2.5 계열이 목록에 보이는지

In [ ]:
if NO_KEY:
    blocked("01_models")
else:
    payload = {"count": len(models), "models": [to_dict(m) for m in models[:60]], "fields_available": sorted(models[0].raw.keys()) if models else []}
    print(save_evidence("gemini", "01_models", payload, "live" if models else "blocked", note="" if models else (adapter.last_error.category if adapter.last_error else "empty"), model=MODEL))

## 2. 모델 호출

- 두 경로가 공존한다.
  - **(A) generateContent** — `POST /v1beta/models/{model}:generateContent` (스트리밍 `:streamGenerateContent`). 요청 최소형 `{"contents":[{"parts":[{"text":"..."}]}]}` + `systemInstruction`, `generationConfig`, `tools`, `safetySettings`, `cachedContent`. SDK `client.models.generate_content(model=, contents=, config=types.GenerateContentConfig(...))` → `resp.text`, `candidates[0].finish_reason`, `usage_metadata`, `model_version`, `response_id`. 공식 문서는 "legacy 지만 fully supported".
  - **(B) Interactions(신규·권장)** — `POST /v1beta/interactions` 요청 `{"model":"gemini-3.8-flash","input":"..."}` (+ `system_instruction`, `generation_config{max_output_tokens, thinking_level…}`, `previous_interaction_id` 서버측 대화 유지, `store` 기본 **true**, `stream`, `background`). SDK `client.interactions.create(model=, input=)` → `interaction.output_text`, `.id`, `.usage`, `.status`. google-genai ≥ 2.3.0 필요.
  - (C) OpenAI 호환 `https://generativelanguage.googleapis.com/v1beta/openai/` (beta, 일부 파라미터 무시).
- 인증: 표준 키(`x-goog-api-key`).
- 주의: **Gemini 4 및 3.6+ 모델은 `temperature`/`top_p`/`top_k` 를 무시**하며 'deprecation warning headers' 를 돌려준다(아래 셀에서 응답 헤더 확인) → 3사 동일조건 비교에 temperature 를 쓸 수 없다. Interactions 는 `store=true` 기본 → 어댑터는 `store=False` 로 보낸다. 두 API 의 usage 필드명이 다르다(§3 표).
- 출처: https://ai.google.dev/api/generate-content · https://ai.google.dev/gemini-api/docs/interactions · https://ai.google.dev/gemini-api/docs/migrate-to-interactions · https://ai.google.dev/gemini-api/docs/text-generation

In [ ]:
# (A) generateContent — 기본 경로. max_output_tokens=64 로 제한 (Free tier 면 $0, Paid 면 $0.001 미만)
if NO_KEY:
    r_gc = None
    print("[BLOCKED: no key]")
else:
    r_gc = adapter.generate(PROMPT, model=MODEL, system="한 문장으로만 답한다.", max_output_tokens=64)  # api 기본값 = "generate_content"
    print(f"status={r_gc.status} latency_ms={r_gc.latency_ms} finish_reason={r_gc.finish_reason} request_id={r_gc.request_id}")
    print("text:", r_gc.text)
    print("error:", to_dict(r_gc.error))

In [ ]:
# (B) Interactions — 같은 프롬프트를 신규 권장 API 로. usage 필드명이 다른 것을 §3 에서 비교한다.
if NO_KEY:
    r_it = None
    print("[BLOCKED: no key]")
else:
    r_it = adapter.generate(PROMPT, model=MODEL, system="한 문장으로만 답한다.", max_output_tokens=64, api="interactions")
    print(f"status={r_it.status} latency_ms={r_it.latency_ms} interaction.status={r_it.finish_reason} id={r_it.request_id}")
    print("text:", r_it.text)
    print("error:", to_dict(r_it.error))

In [ ]:
# 원본 객체/필드 확인 + temperature 무시 여부(3.6+/4): 응답 헤더에 deprecation/warning 류 헤더가 붙는지 본다.
if NO_KEY:
    print("[BLOCKED: no key]")
else:
    print("generate_content raw:", {k: v for k, v in r_gc.raw.items() if k != "config_sent"})
    print("response_headers(화이트리스트):", r_gc.response_headers)
    print("interactions raw:", r_it.raw)
    r_temp = adapter.generate(PROMPT, model=MODEL, max_output_tokens=16, temperature=0.0)  # 3.6+/4 는 무시한다고 문서화됨
    print("temperature=0.0 →", r_temp.status, "warning_headers:", r_temp.raw.get("warning_headers"), "| headers:", r_temp.response_headers)

관찰 메모: (실행 후 채우기) — generate_content vs interactions latency / finish_reason(STOP·MAX_TOKENS) / temperature 전달 시 warning 헤더 이름과 값 / Interactions 응답에 헤더가 없다는 점

In [ ]:
if NO_KEY:
    blocked("02_call")
else:
    ok = r_gc.status == "ok"
    payload = {"generate_content": to_dict(r_gc), "interactions": to_dict(r_it)}
    print(save_evidence("gemini", "02_call", payload, "live" if ok else "blocked", note="" if ok else (r_gc.error.category if r_gc.error else "error"), model=MODEL))

## 3. Token Usage

- 공식 방법: 응답 안의 `usageMetadata`(generateContent) / `interaction.usage`(Interactions). 사전 카운트 `POST /v1beta/models/{model}:countTokens` → `totalTokens` — **과금 안 됨**(billing 문서 "GetTokens requests are not billed"), inference quota 미포함. SDK `client.models.count_tokens(model=, contents=).total_tokens` (SDK 응답 타입은 `total_tokens`, `cached_content_token_count` 만 노출; REST 의 promptTokensDetails 는 없음).
- 필드명 차이(어댑터 `normalize_*_usage` 가 하나로 맞춘다):

| 과금 관점(UsageNormalized) | generateContent `usage_metadata` | Interactions `usage` |
|---|---|---|
| input_tokens (캐시 포함 총입력) | `prompt_token_count` + `tool_use_prompt_token_count` | `total_input_tokens` + `total_tool_use_tokens` |
| cached_input_tokens | `cached_content_token_count` | `total_cached_tokens` |
| output_tokens (thinking 포함) | `candidates_token_count` + `thoughts_token_count` | `total_output_tokens` + `total_thought_tokens` |
| reasoning_tokens | `thoughts_token_count` | `total_thought_tokens` |
| total_tokens | `total_token_count` (= prompt + tool-use + thoughts + candidates) | `total_tokens` |

- 주의: thinking 토큰은 별도 필드지만 **과금은 출력 단가로 합산**("response pricing is the sum of output tokens and thinking tokens"). 모달리티별 분해는 `prompt_tokens_details[]{modality, token_count}`. Gemini API 와 Vertex 간 candidatesTokenCount 에 thinking 포함 여부가 다르다는 이슈 보고가 있어 실측으로 `candidates + thoughts` 와 `total` 의 관계를 확인한다.
- 출처: https://ai.google.dev/api/generate-content · https://ai.google.dev/api/tokens · https://ai.google.dev/gemini-api/docs/tokens · https://ai.google.dev/gemini-api/docs/thinking

In [ ]:
if NO_KEY:
    pre = None
    print("[BLOCKED: no key]")
else:
    pre = adapter.count_tokens(PROMPT, model=MODEL)   # countTokens: 과금 없음
    print("countTokens(사전) total_tokens =", pre if not isinstance(pre, NotSupported) else to_dict(pre))
    print("generate_content usage(정규화):", to_dict(r_gc.usage))
    print("interactions    usage(정규화):", to_dict(r_it.usage))

In [ ]:
# 원본 필드 확인: raw 에 snake_case 원본이 그대로 들어 있다. total == input + output 인지(이중 계산 없음) 검산.
if NO_KEY:
    print("[BLOCKED: no key]")
else:
    for label, r in (("generate_content", r_gc), ("interactions", r_it)):
        if r.usage:
            u = r.usage
            print(f"{label:17s} raw={u.raw}")
            print(f"{'':17s} input+output={u.input_tokens + u.output_tokens} total={u.total_tokens} computed={u.total_is_computed} reasoning={u.reasoning_tokens}")

관찰 메모: (실행 후 채우기) — countTokens 값과 prompt_token_count 의 차이(시스템 프롬프트 포함 여부) / thoughts_token_count 가 0 이 아닌지 / total 검산 결과

In [ ]:
if NO_KEY:
    blocked("03_usage")
else:
    ok = r_gc.status == "ok" and r_gc.usage is not None
    payload = {
        "count_tokens_pre_call": to_dict(pre),
        "generate_content_usage": to_dict(r_gc.usage), "generate_content_raw_usage": r_gc.usage.raw if r_gc.usage else None,
        "interactions_usage": to_dict(r_it.usage), "interactions_raw_usage": r_it.usage.raw if r_it.usage else None,
    }
    print(save_evidence("gemini", "03_usage", payload, "live" if ok else "blocked", note="" if ok else "call failed", model=MODEL))

## 4. Usage 조회

- **공식 API 없음(dashboard_only)** — OpenAI Admin Usage API / Anthropic Usage & Cost Admin API 에 해당하는 엔드포인트가 Gemini API 에는 없다. Admin 키 개념도 없다.
- 공식 경로: AI Studio 대시보드 **https://aistudio.google.com/usage** (로그인, 모델별 요청·토큰) · Rate limits/Quota https://aistudio.google.com/rate-limit?timeRange=last-28-days · Projects 페이지(Tier 확인). 대시보드 백엔드 권한 = `monitoring.timeSeries.list`(Usage), `cloudquotas.quotas.get`(Rate limit), `billing.resourceCosts.get`(Spend).
- 간접(프로그래매틱, **Gemini API 키로는 불가 — Google Cloud OAuth2/ADC + Project Viewer 이상**): Cloud Monitoring `GET https://monitoring.googleapis.com/v3/projects/{project}/timeSeries` 메트릭 **`generativelanguage.googleapis.com/generate_content_usage_output_token_count`**(BETA; labels model, output_modality, thinking_enabled) 와 `quota/generate_content_{free_tier|paid_tier}_input_token_count/usage`, `quota/generate_requests_per_model/usage`(샘플링 후 최대 150초 지연). BigQuery billing export 는 Paid tier + 사전 활성화.
- **주의: Auth key(2026-05-28 이후 신규 키 기본)로 보낸 요청은 "Google Cloud service account usage metrics 에 기록되지 않음"** → Cloud 쪽 간접 집계가 불완전할 수 있음. 결제 데이터는 최대 24h 지연.
- 결론: 3사 공통 집계 경로는 **응답 내 usage 를 호출마다 자체 적재**하는 것뿐(`apicompare.usagelog`). 이 셀은 네트워크를 쓰지 않으므로 evidence 는 LIVE 에서도 `docs` 다.
- 출처: https://ai.google.dev/gemini-api/docs/billing · https://ai.google.dev/gemini-api/docs/rate-limits · https://ai.google.dev/gemini-api/docs/troubleshoot-ai-studio · https://docs.cloud.google.com/monitoring/api/metrics_gcp_d_h · https://ai.google.dev/gemini-api/docs/api-key

In [ ]:
from datetime import datetime, timedelta, timezone
from apicompare import usagelog

now = datetime.now(timezone.utc)
rep = adapter.usage_report(now - timedelta(days=7), now)     # 항상 NotSupported (키 유무 무관)
if NO_KEY:
    print("[BLOCKED: no key]")
print("usage_report →", type(rep).__name__)
print(" reason     :", rep.reason if isinstance(rep, NotSupported) else rep)
print(" alternative:", rep.alternative if isinstance(rep, NotSupported) else "")

In [ ]:
# 대안 확인: 자체 적재 usagelog 집계(§5 에서 append 한 뒤 다시 보면 늘어난다) — 프롬프트/응답 원문·키는 저장하지 않는다
local = usagelog.aggregate(now - timedelta(days=7), now, group_by=("provider", "model"))
print("usagelog(자체 집계):", to_dict(local))
print("대시보드:", "https://aistudio.google.com/usage", "| Monitoring 메트릭: generativelanguage.googleapis.com/generate_content_usage_output_token_count (OAuth, BETA)")

관찰 메모: (실행 후 채우기) — AI Studio Usage 대시보드에서 이 노트북 호출이 보이기까지 걸린 시간 / Auth key vs Standard key 로 Cloud Console 'APIs & Services' 트래픽 그래프에 찍히는지(미해결 질문)

In [ ]:
payload = {"usage_report": to_dict(rep), "usagelog_local": to_dict(local),
           "dashboard": "https://aistudio.google.com/usage",
           "monitoring_metric": "generativelanguage.googleapis.com/generate_content_usage_output_token_count",
           "auth_key_caveat": "Requests authenticated by authorization keys are not recorded in Google Cloud service account usage metrics"}
if NO_KEY:
    print("[BLOCKED: no key]")
    print(save_evidence("gemini", "04_usage_api", payload, "blocked", note="no key", model=MODEL))
else:
    print(save_evidence("gemini", "04_usage_api", payload, "docs", note="API 없음 — 대시보드/Cloud Monitoring(OAuth) 만", model=MODEL))

## 5. 비용 확인

- **비용/단가 엔드포인트 없음** → 로컬 계산: `cost = (prompt − cached) × 입력단가 + cached × 캐시단가(≈입력 10%) + (candidates + thoughts) × 출력단가`. 단가는 공식 pricing 페이지(USD / 1M tokens, "Last updated 2026-10-16") 를 `apicompare.pricing.PRICE_TABLE` 에 as_of·출처와 함께 적어 둔다.
- 2026-10-16 기준 Standard(Paid): `gemini-3.8-flash`·`3.6-flash` **$0.75 / $3.75** → **2027-01-01 부터 $1.50 / $7.50 (2배 인상 예고)**; `gemini-4-argon` $2.00/$10.00 → 2026-12-01 부터 $4/$20(Free tier 없음); 3.5 Flash-Lite $0.30/$2.50; 3.1 Pro Preview $2.00/$12.00(≤200K). Batch/Flex = 50%, Priority = 1.8배. Free tier 는 $0.
- 실비 확인: Paid tier 만 Cloud Billing 리포트(Service 'Gemini API', Group by SKU, 최대 24h 지연)·BigQuery export. Prepay 잔액은 AI Studio. Google Cloud Billing Pricing API 에 Gemini Developer API SKU 가 노출되는지는 미검증(공개 SKU 그룹 'Gen AI Gemini Models' 는 Vertex 소속 — 혼동 주의).
- 주의: Free tier 입력은 제품 개선에 사용될 수 있음(기업 데이터 금지). 2.5 Pro/3.1 Pro 는 200K 기준 2단 요금. Tier 1 은 10분 롤링 $10 지출 한도 → 초과 시 429 RESOURCE_EXHAUSTED.
- 출처: https://ai.google.dev/gemini-api/docs/pricing · https://ai.google.dev/gemini-api/docs/billing · https://docs.cloud.google.com/billing/docs/how-to/reports

In [ ]:
from apicompare import pricing

price = pricing.get_price("gemini", MODEL)
print("단가표 항목:", price)
if NO_KEY:
    cost = None
    print("[BLOCKED: no key]")
else:
    cost = adapter.cost(MODEL, r_gc.usage) if r_gc.usage else None   # 로컬 계산 (CostEstimate | NotSupported)
    print("generate_content 1회 예상 비용:", to_dict(cost))
    if r_it.usage:
        print("interactions     1회 예상 비용:", to_dict(adapter.cost(MODEL, r_it.usage)))

In [ ]:
# 원본 확인: breakdown(input/cached_input/output) · 2027-01-01 인상 예고(effective_from) · 실비 조회는 NotSupported
crep = adapter.cost_report(now - timedelta(days=7), now)
print("cost_report →", type(crep).__name__, "|", crep.reason if isinstance(crep, NotSupported) else crep)
if NO_KEY:
    print("[BLOCKED: no key]")
else:
    if cost is not None and not isinstance(cost, NotSupported):
        print("breakdown:", cost.breakdown, "| price_as_of:", cost.price_as_of, "| effective_from:", cost.effective_from)
        print("note:", cost.note)
        usagelog.append(r_gc, cost, run_id="nb30")   # 자체 적재 (프롬프트·응답 원문·키는 저장하지 않음)
        print("usagelog 적재 →", usagelog.LOG_PATH.name)

관찰 메모: (실행 후 채우기) — 1회 호출 비용(USD) / thoughts 토큰이 비용의 몇 % 인지 / 2027-01-01 인상분 적용 시 비용

In [ ]:
if NO_KEY:
    blocked("05_costs", {"price_table_entry": to_dict(price), "cost_report": to_dict(crep)})
else:
    ok = cost is not None and not isinstance(cost, NotSupported)
    payload = {"local_estimate": to_dict(cost), "price_table_entry": to_dict(price), "cost_report": to_dict(crep)}
    print(save_evidence("gemini", "05_costs", payload, "live" if ok else "blocked", note="" if ok else "no usage", model=MODEL))

## 6. 상태 및 오류 확인

- 헬스체크: `GET /v1beta/models/{model}` (`client.models.get`) — 토큰 소비 없음(메타데이터 조회; 공식 문장은 없음) 또는 `countTokens`(과금 안 됨, billing 문서). 200 이면 키·네트워크·모델 접근 OK, 401/403 이면 인증.
- 오류 포맷 `{"error":{"code":"…","message":"…"}}`, **`error.code` 는 snake_case 문자열**(목록에 없는 코드는 HTTP status 의 snake_case 로 폴백 → 패턴 매칭): 400 `invalid_request`/`failed_precondition`/`parameter_unknown`, 401 `authentication`, **402 `payment_required`(선불 잔액 소진)**, 403 `permission_denied`, 404 `not_found`/`model_not_found`, 429 `rate_limit_exceeded`(RPM/TPM)/`quota_exceeded`(RPD)/`too_many_requests`, 500 `api_error`, **503 `service_unavailable`(과부하)**, 504 `deadline_exceeded`. gRPC 이름(`RESOURCE_EXHAUSTED`, `UNAVAILABLE` …)은 `error.status` 에 병존 → 어댑터는 둘 다 기록.
- SDK 예외: `google.genai.errors.APIError(.code:int, .status, .message, .details)` → `ClientError`(4xx)/`ServerError`(5xx). Interactions 경로는 별도 계층(`APIStatusError.status_code`, `RateLimitError` …) → `error_from_exception` 이 둘 다 처리.
- **레이트리밋 응답 헤더(Retry-After, x-ratelimit-*) 없음**(OpenAI·Anthropic 과의 차이). 한도는 API 키가 아닌 **프로젝트 단위**(RPM/TPM/RPD, RPD 는 태평양시 자정 리셋), 수치는 AI Studio 로그인 후만. 429 바디에 `RetryInfo.retryDelay` 가 온다는 보고는 비공식.
- **SDK 재시도 기본값 함정**: troubleshooting 문서는 "기본 4회 재시도" 라 쓰지만 2.31.0 소스는 `HttpOptions.retry_options=None` → `retry_args(None)` 이 `stop_after_attempt(1)` → **재시도 없음**. `HttpRetryOptions()` 를 넘겼을 때만 attempts=5(초기 포함)·1s→60s·(408,429,500,502,503,504). 아래 셀이 소스를 인용한다. timeout 단위는 **밀리초**; 매우 짧은 timeout 이 500 으로 보고되는 이슈(GitHub #1330) 있음.
- 출처: https://ai.google.dev/gemini-api/docs/api-errors · https://ai.google.dev/gemini-api/docs/troubleshooting · https://ai.google.dev/gemini-api/docs/rate-limits · https://github.com/googleapis/python-genai/blob/main/google/genai/errors.py

In [ ]:
if NO_KEY:
    h = adapter.health()        # ok=False, error.category=AUTH, 네트워크 호출 없음
    print("[BLOCKED: no key]")
    print("health(NO_KEY):", to_dict(h))
else:
    h = adapter.health()
    print(f"health ok={h.ok} status={h.status_code} latency_ms={h.latency_ms} via={h.checked_via} ratelimit_headers={h.ratelimit} error={to_dict(h.error)}")

In [ ]:
# 의도적 오류 재현: 잘못된 키 / 없는 모델 / timeout(1ms) — 더미 키는 짧은 문자열만 쓴다
errors = {}
if NO_KEY:
    print("[BLOCKED: no key]")
else:
    bad_key = get_adapter("gemini", api_key="INVALID-KEY-FOR-TEST", timeout_s=adapter.timeout_s)
    errors["bad_key"] = to_dict(bad_key.generate("ping", model=MODEL, max_output_tokens=8).error)
    errors["bad_model"] = to_dict(adapter.generate("ping", model="model-does-not-exist-123", max_output_tokens=8).error)
    quick = get_adapter("gemini", timeout_s=0.001)   # HttpOptions.timeout=1ms
    errors["timeout_1ms"] = to_dict(quick.generate("ping", model=MODEL, max_output_tokens=8).error)
    for k, v in errors.items():
        print(f"{k:12s} → [{v['category']}] status={v['status_code']} type={v['provider_type']} code={v['provider_code']} exc={v['exception_class']}")
        print(f"{'':12s}   {v['message'][:160]}")
    # 레이트리밋 헤더 부재 확인: §2 응답 헤더에 x-ratelimit-* / retry-after 가 있는가?
    rl = {k: v for k, v in (r_gc.response_headers if r_gc else {}).items() if "ratelimit" in k or k == "retry-after"}
    print("rate-limit 류 헤더:", rl or "없음 (문서대로 — AI Studio 에서만 확인)")

In [ ]:
# 네트워크 불필요 (NO_KEY 에서도 실행됨): ① SDK 재시도 기본값 함정을 소스로 인용 ② 402 payment_required 등 재현 불가 오류의 매핑을 합성 응답으로 확인
import inspect
import httpx
from google.genai import _api_client, errors as genai_errors, types
from apicompare.providers.gemini_adapter import error_from_exception

src = inspect.getsource(_api_client.retry_args)
print("\n".join(l for l in src.splitlines() if "None" in l or "stop_after_attempt" in l or "_RETRY_ATTEMPTS" in l))
print("_RETRY_ATTEMPTS(초기 호출 포함) =", _api_client._RETRY_ATTEMPTS, "| 어댑터 설정 attempts =", adapter.max_retries + 1,
      "| HttpOptions.retry_options 기본 =", types.HttpOptions().retry_options)

def synthetic(status, body):
    resp = httpx.Response(status, json=body, request=httpx.Request("POST", "https://generativelanguage.googleapis.com/v1beta/models/x:generateContent"))
    try:
        genai_errors.APIError.raise_for_response(resp)
    except genai_errors.APIError as e:
        return error_from_exception(e)

for status, body in [
    (402, {"error": {"code": "payment_required", "message": "Prepaid balance exhausted"}}),
    (429, {"error": {"code": "rate_limit_exceeded", "message": "RPM exceeded", "status": "RESOURCE_EXHAUSTED"}}),
    (429, {"error": {"code": "quota_exceeded", "message": "RPD exceeded", "status": "RESOURCE_EXHAUSTED"}}),
    (503, {"error": {"code": 503, "message": "The model is overloaded.", "status": "UNAVAILABLE"}}),
    (401, {"error": {"code": "authentication", "message": "API key not valid", "status": "UNAUTHENTICATED"}}),
]:
    e = synthetic(status, body)
    print(f"{status} {body['error']['code']!s:22s} → {e.category:16s} type={e.provider_type} code={e.provider_code}")

관찰 메모: (실행 후 채우기) — 잘못된 키의 실제 HTTP(400 API_KEY_INVALID 인지 401 인지) / 없는 모델의 code(model_not_found vs not_found) / 1ms timeout 이 TIMEOUT 인지 500 으로 보고되는지(#1330) / 응답 헤더에 x-ratelimit 이 정말 없는지

In [ ]:
payload = {"health": to_dict(h), "intentional_errors": errors or {"note": "키가 없어 의도적 오류 재현은 생략"},
           "sdk_retry_default": {"HttpOptions.retry_options": None, "retry_args(None)": "stop_after_attempt(1) → 재시도 없음", "adapter_attempts": adapter.max_retries + 1},
           "ratelimit_headers_documented": False}
if NO_KEY:
    print("[BLOCKED: no key]")
    print(save_evidence("gemini", "06_errors", payload, "blocked", note="no key", model=MODEL))
else:
    print(save_evidence("gemini", "06_errors", payload, "live" if h.ok else "blocked", note="" if h.ok else (h.error.category if h.error else "health failed"), model=MODEL))

## 7. 호출 예제

- 어댑터 없이 SDK 를 직접 호출하는 최소 예제(10줄 이하) + 같은 요청의 curl. 키는 환경변수 `$GEMINI_API_KEY` 로만 참조한다.
- `pip install -U google-genai` → `export GEMINI_API_KEY=...` → `genai.Client()` 는 환경변수를 자동 인식. 신규 권장 경로는 `client.interactions.create(model=, input=).output_text`.
- Vertex AI(현 'Gemini Enterprise Agent Platform') 는 `genai.Client(vertexai=True, project=, location="global")` + ADC/OAuth, 호스트 aiplatform.googleapis.com, 결제 필수 — API 키 흐름과 다르다. 레거시 `import google.generativeai` 는 금지.
- 출처: https://ai.google.dev/gemini-api/docs/quickstart · https://ai.google.dev/api/generate-content · https://ai.google.dev/gemini-api/docs/interactions · https://ai.google.dev/gemini-api/docs/openai

In [ ]:
if NO_KEY:
    print("[BLOCKED: no key]")
else:
    from google import genai
    from google.genai import types
    client = genai.Client()  # GEMINI_API_KEY 환경변수 자동 인식 — client 객체는 print 하지 않는다
    cfg = types.GenerateContentConfig(max_output_tokens=64)  # 비용 가드: 아래 curl 예제(maxOutputTokens 64)와 동일 요청
    resp = client.models.generate_content(model=MODEL, contents="Explain how AI works in a few words", config=cfg)
    print(resp.text)
    u = resp.usage_metadata
    print(u.prompt_token_count, u.candidates_token_count, u.thoughts_token_count, u.total_token_count, resp.model_version)

In [ ]:
# 동일 요청 curl (generateContent) + Interactions 버전. 키는 $GEMINI_API_KEY 표기만.
CURL_GC = f'''curl -s -X POST "https://generativelanguage.googleapis.com/v1beta/models/{MODEL}:generateContent" \\
  -H "x-goog-api-key: $GEMINI_API_KEY" -H "Content-Type: application/json" \\
  -d '{{"contents":[{{"parts":[{{"text":"Explain how AI works in a few words"}}]}}],"generationConfig":{{"maxOutputTokens":64}}}}' '''
CURL_IT = f'''curl -s -X POST "https://generativelanguage.googleapis.com/v1beta/interactions" \\
  -H "x-goog-api-key: $GEMINI_API_KEY" -H "Content-Type: application/json" \\
  -d '{{"model":"{MODEL}","input":"Explain how AI works in a few words","store":false}}' '''
print(CURL_GC); print(); print(CURL_IT)
if NO_KEY:
    print("\n[BLOCKED: no key]")

관찰 메모: (실행 후 채우기) — SDK 직접 호출 결과가 §2 어댑터 결과와 같은지 / curl 응답의 usageMetadata 키가 camelCase 인지 확인

In [ ]:
if NO_KEY:
    blocked("07_example", {"curl_generate_content": CURL_GC, "curl_interactions": CURL_IT})
else:
    payload = {"sdk_direct": {"text": resp.text, "model_version": resp.model_version, "response_id": resp.response_id,
                              "usage_metadata": resp.usage_metadata.model_dump(exclude_none=True) if resp.usage_metadata else None},
               "curl_generate_content": CURL_GC, "curl_interactions": CURL_IT, "note": "§2 어댑터 호출과 동일 모델·유사 요청"}
    print(save_evidence("gemini", "07_example", payload, "live", model=MODEL))

## 8. 남긴 evidence

`evidence/gemini/*.json` 7개의 배지를 확인한다. NO_KEY 면 전부 `[BLOCKED: no key]` — 키를 받은 뒤 커널 재시작 → Run All 하면 `[LIVE]`/`[DOCS]` 로 바뀐다(§4 는 API 가 없어 LIVE 에서도 `[DOCS]`).

In [ ]:
import pandas as pd
rows = summary("gemini")
print(pd.DataFrame(rows).to_string(index=False))
print("\nMODE:", adapter.mode, "| 다음 단계:", "AI Studio 에서 무료 키 발급 → .env 에 GEMINI_API_KEY 추가 → 커널 재시작 → Run All" if NO_KEY else "docs/providers/gemini.md 의 배지를 이 표로 갱신")